# SetUp

In [ ]:
%%sh
# current_path=$(pwd)
# parent_dir=$(dirname "$current_dir")
export PROJECT_HOME="/home/ivan/Uforse/university_crawl"
echo $PROJECT_HOME
if [ -f "${PROJECT_HOME}/build" ]; then
    echo "Found build in PROJECT_HOME. Sourcing now..."
    . "${PROJECT_HOME}/build"
    echo "build executed successfully."
else
    echo "Error: build not found in PROJECT_HOME."
fi

In [ ]:
import os
import sys
from pprint import pprint

#'/home/ivan/Uforse/university_crawl/fetch_logic'
current_path = os.path.abspath('./')

#'/home/ivan/Uforse/university_crawl'
parent_path = os.path.dirname(current_path)
sys.path.append(parent_path)
import sys
print(sys.executable)
sys.path.append('/home/ivan/Uforse/university_crawl')

In [ ]:
import university_info_generator

from university_info_generator import config
from university_info_generator import UniversityInfoGenerator
from university_info_generator import UniversityBasicInfoType, UniversitySavedDictType, GPTMethodType, HandlerType
from university_info_generator.utility.google_sheet_utility import *
from university_info_generator.utility.save_load_utility import *
from university_info_generator.configs import ALL_ATTRIBUTE_NAME

In [ ]:
import openai
print(openai.__version__)


In [ ]:
# program_attr_df = get_attribute_df(sheet_title="program_attribute_format")
# program_attr_df.head(20)
program_attr_df = pd.read_csv("./program_attr_df.csv")
program_attr_df.head(20)

In [ ]:
program_attr_df.to_csv("program_attr_df.csv")
if 'Unnamed: 0' in program_attr_df.columns:
    program_attr_df = program_attr_df.drop(columns=['Unnamed: 0'])
if "Unnamed: 0.1" in program_attr_df.columns:
    program_attr_df = program_attr_df.drop(columns=['Unnamed: 0.1'])

In [ ]:
program_attr_df.head(20)

In [ ]:
program_attr_df.to_csv("program_attr_df.csv")

In [ ]:
program_attr_df.shape

In [ ]:
import csv
import json
def csv_to_jsonl(csv_file, jsonl_file):
    # Open the CSV file
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)

        # Open the JSONL file for writing
        with open(jsonl_file, 'w', encoding='utf-8') as jsonl_f:
            for row in reader:
                # Convert each row into a JSON object and write to the JSONL file
                for key, value in row.items():
                    if value == "":
                        row[key] = float('nan')  # Convert empty string to NaN
                jsonl_f.write(json.dumps(row) + '\n')



In [ ]:
# Example usage
csv_to_jsonl('program_attr_df.csv', 'program_attr_df.jsonl')

In [ ]:
store_cache("./program_attribute_format.jsonl", get_attribute_dict(sheet_title="program_attribute_format"))

# Main Logic

In [ ]:
uni_program_gen = UniversityInfoGenerator()
uni_program_gen.load_from_file(UniversitySavedDictType.ATTRIBUTE, "./program_attr_df.jsonl")
uni_program_gen.load_from_file(UniversitySavedDictType.GPT_CACHE, "./gpt_cache.jsonl")
uni_program_gen.load_from_file(UniversitySavedDictType.UNIVERSITY_BASIC_INFO, "./university_basic_info_usa.jsonl")
# uni_program_gen.load_from_file(UniversitySavedDictType.UNIVERSITY_INFO, "./university_info_usa.jsonl")

In [ ]:
import pandas as pd

input_df = pd.read_excel("./input.xlsx", sheet_name="links")
input_df.drop(columns=["Unnamed: 9"])
input_df.head()


In [ ]:
from pprint import pp
input_dict = input_df.to_dict(orient="index")
# pp(input_dict)

In [ ]:
attr_dict = get_attribute_dict(sheet_title="program_attribute_format")
# pp(attr_dict)


## fetch logic

In [ ]:
# for key, value in input_dict.items():
#     row = value
#     result = list(map(lambda x: str(x), [
#             row["ON Admission Requirement"],
#             row["IB Admission Requirement"],
#             row["BC Admission Requirement"],
#             row["AP Admission Requirement"],
#             row["Language Requirement"],
#         ]))
#     print(result)
#     break

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

result_dict = {}


# The function to be executed in each thread
def process_row(row, attr_dict, uni_program_gen):
    uni_name = row["University"]
    faculty_name = row["Faculty"]
    program_name = row["Program and link"]
    reference = list(
        map(
            lambda x: str(x) if not pd.isna(x) else "",
            [
                row["ON Admission Requirement"],
                row["IB Admission Requirement"],
                row["BC Admission Requirement"],
                row["AP Admission Requirement"],
                row["Language Requirement"],
            ],
        )
    )
    program_id_ = row["program_id_"]
    row_json = {
        "university_name": uni_name,
        "faculty_name": faculty_name,
        "program_name": program_name,
        "reference": reference,
        "id_": program_id_,
    }

    threads = []

    def process_attribute(attr, value, input_json):
        attr_format = value["attribute_format"]
        attr_prompt = value["attribute_prompt"]
        example = value["example"]
        result = uni_program_gen.get_program_info(
            university_name=uni_name,
            program_name=program_name,
            faculty_name=faculty_name,
            target_attribute=attr,
            format_=attr_format,
            reference=reference,
            _data_example_pair=example,
            _extra_prompt=attr_prompt,
        )
        input_json[attr] = result
        # pp(result)

    for attr, value in attr_dict.items():
        if pd.isna(value["handler"]) or not value["handler"]:
            if attr not in row_json:
                row_json[attr] = ""
            continue
        elif value["handler"] == "GPT_GENERAL":
            thread = threading.Thread(
                target=process_attribute,
                args=(attr, value, row_json),
            )
            threads.append(thread)
            # time.sleep(3)
            thread.start()
    for thread in threads:
        thread.join()

    key = f"({uni_name}, {program_name})"
    result_dict[key] = row_json

In [ ]:
print(len(input_dict))

In [ ]:
for key, value in input_dict.items():
    # print(key, value)
    process_row(value, attr_dict, uni_program_gen)

In [ ]:
print(len(result_dict))

In [ ]:
result_dict

In [ ]:
uni_program_gen.save_to_file(UniversitySavedDictType.GPT_CACHE, "./gpt_cache.jsonl")


In [ ]:
df = pd.DataFrame.from_dict(result_dict, orient="index").reset_index()
df.head(20)

In [ ]:
df.to_csv("./program_output.csv", index=False, encoding="utf-8")